# Topic 4 - Data Quality and Responsible Data Practices
## UTM Library Services Assistant (Individual Project)

This notebook validates, reviews, governs and remediates the corpus handed off from Topic 3.
No embeddings, retrieval or model call (those belong to Topic 7 and Topic 8).

**Topic 3 hand-off**: S17 FAQ 2025 (supplementary, 351 records, native PDF text) and S02 Handbook
2026/2027 (primary authority, 67 records, HTML5 audit + scoped OCR).

**Pipeline here**: combine (418) -> inventory and inspect -> remediate (fix symbols, exclude non-text
pages, re-chunk S17 with overlap, de-space, add topic and authority tags) -> validate -> deduplicate -> coverage
and bias -> report evidence. Final reviewed corpus: 373 chunks.

> Governance: raw source text stays local. Corpus files (`*.jsonl`, `*.pkl`) are produced on disk but
> git-ignored (redistribution not verified). Only public-safe evidence is committed.

## Part A - Requirement-first framing

| Topic 2 requirement | What the prepared data must provide | Quality dimension(s) | Observable check | Decision |
|---|---|---|---|---|
| **FR3 / RAI1** - verifiable reference | Non-empty `source_id`, `source_url`, valid `page` per chunk | Completeness + Accuracy | Rules R2, R3, R4 | Accept / Review / Hold |
| **RAI4** - reviewed version, preserved applicability | `version`, `capture_date`, `approval_status`; not answerable until reviewed | Timeliness + Completeness + Accuracy | `version` present; R7 answering gate | Accept / Review / Hold |

The question is what must be true about the data for FR3 and RAI4 to succeed, not what rule is
easiest to write.

In [ ]:
# Run once in Colab to install dependencies (pandas and matplotlib are preinstalled)
!pip install -q -U pandera langchain-core pyspellchecker
# If a later import still fails, use Runtime > Restart runtime, then run all cells again.

In [ ]:
# Cell 1 - Setup and Topic 3 hand-off (combine the two per-source files)
import os, re, json, pickle, hashlib, collections
import pandas as pd
import matplotlib.pyplot as plt
try:
    import pandera.pandas as pa      # pandera >= 0.18
except ImportError:
    import pandera as pa             # older pandera fallback
from langchain_core.documents import Document

os.makedirs('data', exist_ok=True)   # output folders (Colab /content has neither by default)
os.makedirs('docs', exist_ok=True)

# --- Input location (Week-3-style): put the two Topic 3 files in this Google Drive folder ---
# They then load automatically every run - no manual upload. Change the path if your folder differs.
INPUT_DIR = '/content/drive/MyDrive/AE_Labs/AE_Lab4/inputs'
try:
    from google.colab import drive
    drive.mount('/content/drive')          # click to authorise once per session
except Exception:
    pass                                    # not on Colab; fall back to local/uploaded files

def find(names):
    roots = [INPUT_DIR, '.', 'data', '/content', '/content/data', '/content/drive/MyDrive',
             'week3_project_artifacts/week3_project/data']
    for r in roots:
        for n in names:
            if os.path.exists(os.path.join(r, n)): return os.path.join(r, n)
    raise FileNotFoundError(
        "Topic 3 file not found (" + " or ".join(names) + "). In Colab, upload "
        "'prepared_text.jsonl' and 'prepared_handbook_text.jsonl' to /content "
        "(Files panel on the left > Upload), or place them in your Drive folder "
        "INPUT_DIR and re-run after Drive is mounted.")

load = lambda p: [json.loads(l) for l in open(p, encoding='utf-8') if l.strip()]
s17 = load(find(['processed/prepared_text.jsonl', 'prepared_text.jsonl']))
s02 = load(find(['pipeline_v1/handbook_html5_ocr/prepared_handbook_text.jsonl', 'prepared_handbook_text.jsonl']))
raw = s17 + s02
chunks_raw = [Document(page_content=r.get('text','') or '',
                       metadata={**{k:v for k,v in r.items() if k!='text'}, 'source': r.get('source_id',''), 'chunk_id': i})
              for i, r in enumerate(raw)]
print('Combined input:', len(chunks_raw), '| by source:', dict(collections.Counter(c.metadata['source'] for c in chunks_raw)))

In [ ]:
# Cell 2 - Quality inventory (helper reused later on the cleaned corpus)
def inventory(chunk_list):
    rows = []
    for i, c in enumerate(chunk_list):
        t = c.page_content or ''; m = c.metadata or {}; st = t.strip()
        rows.append({'chunk_id': i, 'record_id': m.get('record_id',''), 'source': str(m.get('source','') or ''),
            'page': m.get('page'), 'char_count': len(st),
            'missing_text': len(st)==0, 'missing_source': str(m.get('source','') or '').strip()=='',
            'missing_url': str(m.get('source_url','') or '').strip()=='', 'missing_page': m.get('page') in (None,'',0),
            'has_bad_char': ('\x00' in t) or ('\ufffd' in t),
            'approved_for_answering': bool(m.get('approved_for_answering', False)),
            'manual_review_queue': bool(m.get('manual_review_queue', False)),
            '_preview': re.sub(r'\s+',' ', st)[:40]})
    return pd.DataFrame(rows)

df_raw = inventory(chunks_raw)
print('Missing text/source/url/page:', int(df_raw.missing_text.sum()), int(df_raw.missing_source.sum()),
      int(df_raw.missing_url.sum()), int(df_raw.missing_page.sum()))
print('Replacement char:', int(df_raw.has_bad_char.sum()), '| under 50 chars:', int((df_raw.char_count<50).sum()),
      '| S02 review-queued:', int(df_raw.manual_review_queue.sum()))

In [ ]:
# Cell 3 - Manual inspection
look = df_raw.sort_values('char_count').head(6)['chunk_id'].tolist() + \
       df_raw[df_raw.has_bad_char]['chunk_id'].tolist() + df_raw[df_raw.manual_review_queue]['chunk_id'].tolist()
for idx in look:
    c = chunks_raw[int(idx)]
    print('---', c.metadata.get('source'), c.metadata.get('record_id'), '| page', c.metadata.get('page'),
          '| chars', len(c.page_content.strip()), '---'); print(repr(c.page_content.strip()[:120]))

### A3 - Manual inspection and rendered-page review

| Observation | Dimension / concern | Evidence | Decision |
|---|---|---|---|
| S02 pages 17, 24, 27 are OCR garble | Accuracy (measurement) | **Rendered-page review**: full-page section dividers ("MEMBERSHIPS", "LOAN ELIGIBILITY", "FINES") over photos; no text; garble persists at 300 DPI | Exclude by design, log |
| 20 S17 chunks are 3-49 chars (`94)`, `Lumpur`, `ties.`) | Completeness (fragments) | Tails of sentences split by the 500-char, overlap=0 chunker | Re-chunk with overlap |
| 7 S17 pages are cover / section dividers | Completeness (navigation) | Pages 1, 2, 30, 42, 51, 63, 71 hold only a section label | Drop as navigation; keep label as `faq_section` |
| 4 S17 chunks show `U+FFFD` in `Map location` lines | Accuracy (encoding) | Map-pin glyph lost; URL survives | Strip symbol |
| S17-p032 renewal statement repeats | Uniqueness | Same text twice on page 32 | Remove in reconstruction |

**Key S02 finding.** The handbook is image-only, so all S02 text is OCR. The garbled chunks are not
lost policy: they are decorative dividers. The real policy pages (for example page 20) OCR cleanly.

**Reflection.** The hardest issue to express as a numeric rule is applicability (RAI4): an S17 2025
statement can be well-formed and cited yet superseded by the 2026/2027 handbook. A rule confirms the
`version` field exists but cannot decide current applicability; that needs source precedence plus
human review.

In [ ]:
# Cell 4 - Remediation: fix symbols, exclude non-text pages, re-chunk S17 with overlap, add tags
BOILER = 'FREQUENTLY ASK QUESTION (FAQ) / 2025'
def faq_section(p):
    return ('COVER' if p==1 else 'GENERAL' if p<=29 else 'SERVICES' if p<=41 else 'FACILITIES'
            if p<=50 else 'COLLECTIONS' if p<=62 else 'ACQUISITIONS' if p<=70 else 'OTHERS')
NAV_PAGES = {1,2,30,42,51,63,71}          # cover + 6 FAQ section dividers (no Q&A content)
DIVIDER_PAGES = {17,24,27}                 # S02 section-divider photo pages (rendered-page review)
def hb_section(p):
    return ('MEMBERSHIPS' if 18<=p<=23 else 'LOAN_ELIGIBILITY' if 25<=p<=26 else 'FINES' if 28<=p<=29
            else 'GENERAL_OR_SERVICES')

# dictionary de-spacing: merge a broken split (a b) only when a+b is a word and a or b is not
try:
    from spellchecker import SpellChecker   # Colab: !pip install -q pyspellchecker
    _SP = SpellChecker()
    def despace(t):
        def r(m):
            a, b = m.group(1), m.group(2); j = a + b
            if j.lower() in _SP and (a.lower() not in _SP or b.lower() not in _SP) and len(b) >= 2: return j
            return m.group(0)
        return re.sub(r'\b([A-Za-z]{1,8}) ([A-Za-z]{2,8})\b', r, t)
except Exception:
    def despace(t): return t

def rechunk(text, size=500, overlap=80, min_last=100):
    text = re.sub(r'\ufffd+', '', text)
    paras, seen, uniq = [p.strip() for p in re.split(r'\n{2,}', text) if p.strip()], set(), []
    for p in paras:                        # drop exact-duplicate paragraphs within the page
        k = re.sub(r'\s+',' ', p)
        if k not in seen: seen.add(k); uniq.append(p)
    units = []
    for p in uniq:
        if len(p) <= size: units.append(p); continue
        buf = ''
        for s in re.split(r'(?<=[.!?])\s+', p):
            if len(buf)+len(s)+1 <= size: buf = (buf+' '+s).strip()
            else:
                if buf: units.append(buf)
                if len(s) > size:
                    for i in range(0, len(s), size-overlap): units.append(s[i:i+size])
                    buf = ''
                else: buf = s
        if buf: units.append(buf)
    chunks, cur = [], ''
    for u in units:
        if len(cur)+len(u)+2 <= size: cur = (cur+'\n\n'+u).strip()
        else:
            if cur: chunks.append(cur)
            cur = u
    if cur: chunks.append(cur)
    while len(chunks) >= 2 and len(chunks[-1]) < min_last:
        chunks[-2] = chunks[-2]+'\n\n'+chunks[-1]; chunks.pop()
    if overlap > 0 and len(chunks) > 1:    # sentence-ish overlap: prepend tail of previous chunk
        out = [chunks[0]]
        for i in range(1, len(chunks)):
            tail = chunks[i-1][-overlap:]; sp = tail.find(' ')
            out.append(((tail[sp+1:] if sp!=-1 else tail)+' '+chunks[i]).strip())
        chunks = out
    return [c for c in chunks if c.strip()]

excluded = []
# S17 re-chunk per page
by_page = collections.defaultdict(list)
for r in s17: by_page[r['page']].append(r)
s17_docs = []
for page in sorted(by_page):
    recs = sorted(by_page[page], key=lambda r: r['record_id'])
    base = {k:v for k,v in recs[0].items() if k!='text'}
    if page in NAV_PAGES:
        excluded.append({'record_id': recs[0]['record_id'], 'source':'S17', 'page':page,
            'reason':'FAQ navigation page (cover or section divider), no Q&A content',
            'evidence':'section label only', 'faq_section':faq_section(page)}); continue
    page_text = '\n\n'.join([(r.get('text') or '').replace(BOILER,'').strip() for r in recs if (r.get('text') or '').strip()])
    for ci, ct in enumerate(rechunk(page_text), 1):
        ct = despace(ct)
        m = dict(base); m.update({'source':'S17', 'faq_section':faq_section(page),
            'record_id':'S17-p%03d-c%03d'%(page,ci), 'authority_rank':2, 'authority_tier':'supplementary',
            'processing':'re-chunked v2: paragraph+sentence-aware size=500 overlap=80; trailing-merge'})
        m.pop('chunk_id', None)
        s17_docs.append(Document(page_content=ct, metadata=m))

# S02: exclude dividers, add tags
s02_docs = []
for r in s02:
    p = r.get('page')
    if p in DIVIDER_PAGES:
        excluded.append({'record_id':r.get('record_id'), 'source':'S02', 'page':p,
            'reason':'section-divider photo page (no policy text), confirmed by rendered-page review',
            'evidence':'title banner over library photo; OCR garble persists at 300 DPI', 'faq_section':''}); continue
    m = {k:v for k,v in r.items() if k!='text'}
    m.update({'source':'S02', 'handbook_section':hb_section(p), 'authority_rank':1, 'authority_tier':'primary'})
    s02_docs.append(Document(page_content=re.sub(r'\ufffd+','', r.get('text') or ''), metadata=m))

chunks = s17_docs + s02_docs
os.makedirs('docs', exist_ok=True)
pd.DataFrame(excluded).to_csv('docs/week4_excluded_records.csv', index=False)
df = inventory(chunks)
print('S17 re-chunked:', len(s17_docs), '| S02:', len(s02_docs), '| excluded:', len(excluded),
      '| under 50 now:', int((df.char_count<50).sum()), '| bad char:', int(df.has_bad_char.sum()))

In [ ]:
# Cell 5 - Validation rules (Pandera) + governance-safe register
MIN_LEN = 50
schema = pa.DataFrameSchema({
    'chunk_id':   pa.Column(int, checks=pa.Check.ge(0), nullable=False),
    'source':     pa.Column(str, checks=pa.Check(lambda s: s.str.strip().str.len()>0, error='R2'), nullable=False),
}, strict=False)
try: schema.validate(df, lazy=True); print('Pandera R2: pass')
except pa.errors.SchemaErrors as e: print(e.failure_cases[['column','check','index']])

def collect(mask, rule, req, sev, cls):
    return [{'rule':rule,'requirement':req,'severity':sev,'classification':cls,'chunk_id':int(r.chunk_id),
             'record_id':r.record_id,'source':r.source,'page':r.page,'char_count':int(r.char_count),'preview':r._preview}
            for _, r in df[mask].iterrows()]
reg = (collect(df.missing_text,'R1 text_present','FR1/FR2','HARD_FAIL','missing') +
       collect(df.missing_source,'R2 source_present','FR3/RAI1','HARD_FAIL','missing') +
       collect(df.missing_url,'R3 source_url_present','RAI4','HARD_FAIL','missing') +
       collect(df.missing_page,'R4 page_locator_present','FR3','HARD_FAIL','missing') +
       collect(df.has_bad_char,'R5 no_replacement_char','RAI1 accuracy','REVIEW','corrupted/noisy') +
       collect(df.char_count<MIN_LEN,'R6 min_useful_length','FR2 completeness','REVIEW','short-heading') +
       collect(df.approved_for_answering,'R7 answering_gate','RAI1/RAI4','HARD_FAIL','governance'))
fail_df = pd.DataFrame(reg); fail_df.to_csv('docs/week4_validation_failures.csv', index=False)
for rule in ['R1 text_present','R2 source_present','R3 source_url_present','R4 page_locator_present',
             'R5 no_replacement_char','R6 min_useful_length','R7 answering_gate']:
    print(f'{rule:26s}: {int((fail_df.rule==rule).sum()) if len(fail_df) else 0}')

### A4 - Validation rules

| Rule | Condition | Requirement / failure | Flagged | Decision |
|---|---|---|---|---|
| R1 text_present | `char_count > 0` | No evidence text (FR1/FR2) | 0 | Hard fail |
| R2 source_present | `source_id != ""` | Untraceable (FR3/RAI1) | 0 | Hard fail |
| R3 source_url_present | `source_url != ""` | No provenance link (RAI4) | 0 | Hard fail |
| R4 page_locator_present | `page` valid | No locator (FR3) | 0 | Hard fail |
| R5 no_replacement_char | no `U+FFFD` | Corruption | 0 (fixed) | Resolved |
| R6 min_useful_length | `char_count >= 50` | Fragment too short | 11 | Review (complete headings, carried into next chunk via overlap) |
| R7 answering_gate | all K2 `approved_for_answering == False` | Answerable before review | 0 | Hard fail - confirmed |

All hard-fail rules pass. R5 resolved; R6 now flags complete section/page headings, not broken tails.

In [ ]:
# Cell 6 - Exact dedup + save cleaned corpus
seen, unique, dup = {}, [], []
for c in chunks:
    norm = re.sub(r'\s+',' ', (c.page_content or '').strip())
    h = hashlib.sha256(norm.encode()).hexdigest()
    if h not in seen: seen[h] = c.metadata.get('record_id'); unique.append(c)
    else: dup.append({'hash':h[:16],'duplicate_record':c.metadata.get('record_id'),
                      'duplicate_source':c.metadata.get('source'),'first_record':seen[h],
                      'char_count':len(norm),'preview':norm[:40]})
for i,c in enumerate(unique): c.metadata['chunk_id'] = i
print('Before:', len(chunks), '| after dedup:', len(unique), '| duplicates:', len(dup))
pd.DataFrame(dup).to_csv('docs/week4_duplicate_review.csv', index=False)
with open('data/chunks_clean.pkl','wb') as f: pickle.dump(unique, f)
with open('data/prepared_chunks_clean.jsonl','w',encoding='utf-8') as f:
    for i,c in enumerate(unique): f.write(json.dumps({'chunk_id':i,'text':c.page_content,'metadata':c.metadata},ensure_ascii=False)+'\n')
print('Saved cleaned corpus (kept local by governance).')

In [ ]:
# Cell 7 - Coverage + authority tiers (Part B2)
src = collections.Counter(c.metadata.get('source') for c in unique)
cov = pd.DataFrame([{'source':s,'chunks':n,'percent':round(n/len(unique)*100,1)} for s,n in src.items()]).sort_values('chunks',ascending=False)
print(cov.to_string(index=False))
plt.figure(figsize=(7,4)); bars = plt.bar(cov['source'].astype(str), cov['chunks'], color=['#4C78A8','#F58518'])
for b,n,p in zip(bars, cov['chunks'], cov['percent']): plt.text(b.get_x()+b.get_width()/2, n+1.5, f'{n} ({p}%)', ha='center', va='bottom', fontsize=9)
plt.ylabel('Number of chunks (cleaned corpus)'); plt.ylim(0, max(cov['chunks'])*1.18)
plt.title('Chunk Coverage by Source - UTM Library Services Assistant')
plt.tight_layout(); plt.savefig('docs/week4_coverage_by_source.png', dpi=130); plt.show()
print('authority_rank:', dict(collections.Counter(c.metadata.get('authority_rank') for c in unique)))
print('S17 faq_section:', dict(collections.Counter(c.metadata.get('faq_section') for c in unique if c.metadata.get('source')=='S17')))

In [ ]:
# Cell 7b - Lexical near-duplicate / cross-source overlap + S02 task coverage (embeddings deferred to Topic 7)
def shingles(t, k=5):
    w = re.findall(r'[a-z0-9]+', t.lower())
    return set(tuple(w[i:i+k]) for i in range(max(0, len(w)-k+1))) if len(w) >= k else {tuple(w)}
sh = [shingles(c.page_content) for c in unique]; near = []
for a in range(len(unique)):
    for b in range(a+1, len(unique)):
        if not sh[a] or not sh[b]: continue
        inter = len(sh[a] & sh[b])
        if not inter: continue
        jac = inter / len(sh[a] | sh[b]); same = unique[a].metadata['source'] == unique[b].metadata['source']
        ra, rb = unique[a].metadata['record_id'], unique[b].metadata['record_id']
        adj = same and ra.rsplit('-c',1)[0] == rb.rsplit('-c',1)[0]
        if jac >= (0.5 if same else 0.35) and not adj:
            near.append({'jaccard': round(jac,3), 'type': 'within-source' if same else 'cross-source', 'record_a': ra, 'record_b': rb})
near = sorted(near, key=lambda x: -x['jaccard'])
pd.DataFrame(near).to_csv('docs/week4_near_duplicate_review.csv', index=False)
print('near-duplicate pairs:', len(near), '| cross-source:', sum(1 for n in near if n['type']=='cross-source'))
def s02pages(kws): return sorted({c.metadata['page'] for c in unique if c.metadata['source']=='S02' and any(k in c.page_content.lower() for k in kws)})
print('S02 membership   :', s02pages(['member','registration','activate']))
print('S02 borrow/renew :', s02pages(['borrow','loan','renew','return','reserv']))
print('S02 fines/rules  :', s02pages(['fine','overdue','penalt','charge','rule']))

### B2 - Coverage interpretation

- S17 (supplementary) = 311 chunks / 83.4 percent, tagged into 6 FAQ sections; S02 (primary) = 62 /
  16.6 percent across 19 policy pages. Grouping is actual metadata, not a proxy.
- **Why S02 is only about 16 percent:** (1) S02 native extraction failed (FP-01), so only scoped
  policy pages were OCR-captured; (2) the handbook is photo-heavy, so many of its 76 pages are
  non-text; (3) S17 is a large 85-page FAQ with a text layer. The share reflects extraction difficulty
  and document shape, not low policy importance. The handbook stays the primary authority.

### B3 - Bias risks, propagation and mitigation

| Bias type | Evidence | Origin | Downstream impact | Mitigation | Test (Topic 8) |
|---|---|---|---|---|---|
| **Representation** | S02 primary 16.6% vs S17 83.4%; S02 = 19 pages | Engineering: FP-01 + photo-heavy handbook | Count-blind retrieval may rank FAQ above the handbook | `authority_rank` lets retrieval prefer S02; confirm policy completeness | Primary-vs-supplementary retrieval hit-rate on 30 queries |
| **Measurement** | Handbook image-only; S02 tables/numbers unverified | Measurement: OCR approximates the page | Wrong facts (hours, fines) if an unverified table is used | Keep S02 blocked; spot rendered-page check | OCR text vs rendered page |
| **Historical** | S17 2025 vs S02 2026/2027 | Source condition | Stale policy returned as current | Source precedence + timeliness metadata; hold conflicts | Controlled conflict case (E8/RAI4) |

**Reflection.** The representation risk is an engineering condition and is now partly mitigated in the
data by `authority_rank`; the measurement risk is contained by the answering gate and rendered-page
checks. Both are data-level signals; fairness is evaluated in Topic 8.

### B3 - Responsible-data governance review

| Governance question | Evidence | Status | Action / owner |
|---|---|---|---|
| Owner, version/date, provenance traceable? | `source_id`, `source_url`, `source_sha256`, `page`, `version`, `capture_date` | OK | Keep provenance on later artefacts |
| Rights/access terms known and appropriate? | Publicly viewable; redistribution not verified | To verify | Confirm reuse basis before any public release |
| Personal/sensitive data unnecessary? | Public policy text only | No | None |
| Consent/collection basis? | Public institutional documents | N/A | Record N/A with reason |
| Update/retention/removal defined? | Source-precedence rule + versioned sources | OK (partial) | Assign an update owner |

> Governance boundary: publicly accessible is not public domain. Snapshots and extracted text stay
> local; rights basis marked to verify.

## Part C - Report and quality gate

Full report: `docs/week4_data_quality_report.md`.

**GO WITH CONDITIONS.** Data-level defects are resolved: S17 re-chunked with overlap, navigation and
divider pages excluded with evidence, symbols and duplicate removed, topic tags added, and source
precedence made operational via `authority_rank`. The one remaining step is human: spot-verify S02 OCR
tables and numbers against the rendered handbook page before S02 is approved. S02 300 DPI re-OCR was
tested and held (local engine degraded text; defer to Colab tesseract 5.3.4). Rights remain to verify.

In [ ]:
# Cell 8 - Verify outputs
cp = pickle.load(open('data/chunks_clean.pkl','rb'))
cj = [json.loads(l) for l in open('data/prepared_chunks_clean.jsonl', encoding='utf-8') if l.strip()]
print('chunks_clean.pkl:', len(cp), '| prepared_chunks_clean.jsonl:', len(cj))
assert len(cp)==len(cj), 'mismatch'
print('Counts match. Reviewed Topic 4 corpus ready for Topic 7.')
print('Answerable now:', sum(1 for c in cp if c.metadata.get('approved_for_answering')), '(expected 0)')

In [ ]:
# Cell 9 - Save this run's outputs to Google Drive (like Week 3; private, not committed)
import shutil, glob
OUTPUT_DIR = '/content/drive/MyDrive/AE_Labs/AE_Lab4/outputs'   # change to your folder if different
try:
    os.makedirs(OUTPUT_DIR + '/data', exist_ok=True)
    os.makedirs(OUTPUT_DIR + '/docs', exist_ok=True)
    saved = []
    for f in glob.glob('data/*.pkl') + glob.glob('data/*.jsonl'):
        shutil.copy(f, OUTPUT_DIR + '/data/'); saved.append(f)
    for f in glob.glob('docs/week4_*'):
        shutil.copy(f, OUTPUT_DIR + '/docs/'); saved.append(f)
    print('Saved', len(saved), 'files to', OUTPUT_DIR)
    for s in sorted(saved): print('  ', s)
    print('Note: the corpus files contain source text - keep this Drive folder private.')
except Exception as e:
    print('Drive not mounted / not on Colab - skipped Drive save:', e)
    print('Outputs are still in /content/data and /content/docs (download from the Files panel).')
